# Supplement results with missing columns

In [31]:
from specvae import utils
import os
import pandas as pd
import numpy as np
from specvae.model import BaseModel

In [32]:
# config_names = ['reg_layer_config', 'reg_input_columns']
config_names = ['clf_layer_config', ['clf_config', 'class_subset'], 'clf_input_columns']
base_path = utils.get_project_path() / '.model' / 'MoNA' / 'betavae_clf'
insert_after_columns = ['layer_config', 'input_columns', 'input_columns'] # empty value means no preference where to insert
paths = [
    base_path / 'experiment01_pfi.csv',
    base_path / 'experiment02_pfi.csv',
    base_path / 'experiment03_pfi.csv',
    base_path / 'experiment04_pfi.csv',
    # base_path / 'experiment04.csv',
]

In [33]:
device, cpu = utils.device(use_cuda=False)

Device in use:  cpu


In [34]:
def load_model(path):
    model_path = os.path.join(path, 'model.pth')
    model = BaseModel.load(model_path, device)
    model.eval()
    return model

In [35]:
def get_config(base_path, full_model_name, name):
    path = base_path / full_model_name
    model = load_model(str(path))
    if isinstance(name, str):
        return model.config[name]
    else:
        cfg = model.config
        for n in name:
            cfg = cfg[n]
        return cfg

In [36]:
for path in paths:
    print("Process file:", str(path))
    df = pd.read_csv(path, index_col=0)
    for config_name, insert_after_column in zip(config_names, insert_after_columns):
        column_name = config_name if isinstance(config_name, str) else config_name[-1]
        if len(insert_after_column) > 0:
            loc = df.columns.get_loc(insert_after_column) + 1
            df.insert(loc, column_name, '')
        df[column_name] = df['full_model_name'].apply(lambda fmn: get_config(base_path, fmn, config_name))
    # Save results in a copy of csv file:
    path_, filename = os.path.split(path)
    newfilename = '%s_sup.csv' % os.path.splitext(filename)[0]
    new_filepath = os.path.join(path_, newfilename)
    df.to_csv(new_filepath)
    print("Processing complete! File saved in", new_filepath)
print("DONE!")

Process file: /home/krzyja/Workspace/SpecVAE/.model/HMDB/betavae_clf/experiment01_pfi.csv
Processing complete! File saved in /home/krzyja/Workspace/SpecVAE/.model/HMDB/betavae_clf/experiment01_pfi_sup.csv
Process file: /home/krzyja/Workspace/SpecVAE/.model/HMDB/betavae_clf/experiment02_pfi.csv
Processing complete! File saved in /home/krzyja/Workspace/SpecVAE/.model/HMDB/betavae_clf/experiment02_pfi_sup.csv
DONE!
